In [10]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.over_sampling import RandomOverSampler


def run_tree_model(x_train_path, x_test_path, y_train_path, y_test_path, upsample=False):
    # 1. read data
    X_train = pd.read_csv(x_train_path)
    X_test = pd.read_csv(x_test_path)
    y_train = pd.read_csv(y_train_path)["y"]
    y_test = pd.read_csv(y_test_path)["y"]

    # 2. define CV setting
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # 3. define parameter grid
    param_grid = {
        "max_depth": [3, 5, 10],
        "min_samples_split": [2, 5, 10],
        "min_weight_fraction_leaf": [0.0, 0.05, 0.1],
        "min_impurity_decrease": [0.0, 0.01, 0.05]
    }

    # 4. tune model on training data
    if upsample:
        upsampler = RandomOverSampler(random_state=42)
        X_train_tune, y_train_tune = upsampler.fit_resample(
            X_train,
            y_train
        )
    else:
        X_train_tune, y_train_tune = X_train, y_train

    tree_base = DecisionTreeClassifier(
        random_state=42
    )

    tree_search = GridSearchCV(
        estimator=tree_base,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )

    tree_search.fit(X_train_tune, y_train_tune)
    best_params = tree_search.best_params_

    # 5. define tuned model
    tree_model = DecisionTreeClassifier(
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        min_weight_fraction_leaf=best_params["min_weight_fraction_leaf"],
        min_impurity_decrease=best_params["min_impurity_decrease"],
        random_state=42
    )

    # 6. fivefold cross-validation with tuned model
    cv_scores = {
        "AUC": [],
        "Precision": [],
        "Recall": [],
        "Accuracy": [],
        "F1": []
    }

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]

        X_val_fold = X_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]

        if upsample:
            # Apply random over-sampling only to the training fold
            upsampler = RandomOverSampler(random_state=42)
            X_train_fold, y_train_fold = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

        tree_model.fit(X_train_fold, y_train_fold)

        y_val_pred = tree_model.predict(X_val_fold)
        y_val_prob = tree_model.predict_proba(X_val_fold)[:, 1]

        cv_scores["AUC"].append(
            roc_auc_score(y_val_fold, y_val_prob)
        )
        cv_scores["Precision"].append(
            precision_score(y_val_fold, y_val_pred, zero_division=0)
        )
        cv_scores["Recall"].append(
            recall_score(y_val_fold, y_val_pred, zero_division=0)
        )
        cv_scores["Accuracy"].append(
            accuracy_score(y_val_fold, y_val_pred)
        )
        cv_scores["F1"].append(
            f1_score(y_val_fold, y_val_pred, zero_division=0)
        )

    cv_metrics_df = pd.DataFrame({
        "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
        "Mean CV Score": [
            np.mean(cv_scores["AUC"]),
            np.mean(cv_scores["Precision"]),
            np.mean(cv_scores["Recall"]),
            np.mean(cv_scores["Accuracy"]),
            np.mean(cv_scores["F1"])
        ],
        "CV Std": [
            np.std(cv_scores["AUC"], ddof=1),
            np.std(cv_scores["Precision"], ddof=1),
            np.std(cv_scores["Recall"], ddof=1),
            np.std(cv_scores["Accuracy"], ddof=1),
            np.std(cv_scores["F1"], ddof=1)
        ]
    }).round(3)

    # 7. fit final model on full training data
    if upsample:
        upsampler = RandomOverSampler(random_state=42)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train,
            y_train
        )
        tree_model.fit(X_train_final, y_train_final)
    else:
        tree_model.fit(X_train, y_train)

    # 8. predict on independent test set
    y_pred = tree_model.predict(X_test)
    y_prob = tree_model.predict_proba(X_test)[:, 1]

    # 9. test set metrics
    test_metrics_df = pd.DataFrame({
        "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
        "Test Score": [
            roc_auc_score(y_test, y_prob),
            precision_score(y_test, y_pred, zero_division=0),
            recall_score(y_test, y_pred, zero_division=0),
            accuracy_score(y_test, y_pred),
            f1_score(y_test, y_pred, zero_division=0)
        ]
    }).round(3)

    return {
        "best_params": best_params,
        "cv_metrics": cv_metrics_df,
        "test_metrics": test_metrics_df
    }

In [11]:
def make_cv_summary(results_dict):
    rows = []

    for model_name, result in results_dict.items():
        cv_df = result["cv_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            mean_value = cv_df.loc[
                cv_df["Metric"] == metric,
                "Mean CV Score"
            ].values[0]

            cv_std = cv_df.loc[
                cv_df["Metric"] == metric,
                "CV Std"
            ].values[0]

            # Standard error across five folds
            cv_se = cv_std / np.sqrt(5)

            row[metric] = f"{mean_value:.3f} ({cv_se:.3f})"

        rows.append(row)

    return pd.DataFrame(rows)


def make_test_summary(results_dict):
    rows = []

    for model_name, result in results_dict.items():
        test_df = result["test_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            test_value = test_df.loc[
                test_df["Metric"] == metric,
                "Test Score"
            ].values[0]

            row[metric] = round(test_value, 3)

        rows.append(row)

    return pd.DataFrame(rows)

In [12]:
# Run classification tree models for Model 1
result_1 = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_1.csv",
    x_test_path="../data/model_inputs/X_test_model_1.csv",
    y_train_path="../data/model_inputs/y_train_model_1.csv",
    y_test_path="../data/model_inputs/y_test_model_1.csv",
    upsample=False
)

result_1a = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_1a.csv",
    x_test_path="../data/model_inputs/X_test_model_1a.csv",
    y_train_path="../data/model_inputs/y_train_model_1a.csv",
    y_test_path="../data/model_inputs/y_test_model_1a.csv",
    upsample=True
)

result_1b = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_1b.csv",
    x_test_path="../data/model_inputs/X_test_model_1b.csv",
    y_train_path="../data/model_inputs/y_train_model_1b.csv",
    y_test_path="../data/model_inputs/y_test_model_1b.csv",
    upsample=True
)

In [13]:
# Run classification tree models for Model 2
result_2 = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_2.csv",
    x_test_path="../data/model_inputs/X_test_model_2.csv",
    y_train_path="../data/model_inputs/y_train_model_2.csv",
    y_test_path="../data/model_inputs/y_test_model_2.csv",
    upsample=False
)

result_2a = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_2a.csv",
    x_test_path="../data/model_inputs/X_test_model_2a.csv",
    y_train_path="../data/model_inputs/y_train_model_2a.csv",
    y_test_path="../data/model_inputs/y_test_model_2a.csv",
    upsample=True
)

result_2b = run_tree_model(
    x_train_path="../data/model_inputs/X_train_model_2b.csv",
    x_test_path="../data/model_inputs/X_test_model_2b.csv",
    y_train_path="../data/model_inputs/y_train_model_2b.csv",
    y_test_path="../data/model_inputs/y_test_model_2b.csv",
    upsample=True
)

In [14]:
# Store Model 1 results
tree_model1_results = {
    "Tree Model 1": result_1,
    "Tree Model 1a": result_1a,
    "Tree Model 1b": result_1b
}

# Store Model 2 results
tree_model2_results = {
    "Tree Model 2": result_2,
    "Tree Model 2a": result_2a,
    "Tree Model 2b": result_2b
}


# Print best parameters
print("\n===== Best Parameters (tree model 1) =====")
for model_name, result in tree_model1_results.items():
    print(f"\n{model_name}")
    print(result["best_params"])

print("\n===== Best Parameters (tree model 2) =====")
for model_name, result in tree_model2_results.items():
    print(f"\n{model_name}")
    print(result["best_params"])


===== Best Parameters (tree model 1) =====

Tree Model 1
{'max_depth': 10, 'min_impurity_decrease': 0.0, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0}

Tree Model 1a
{'max_depth': 10, 'min_impurity_decrease': 0.0, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0}

Tree Model 1b
{'max_depth': 10, 'min_impurity_decrease': 0.0, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0}

===== Best Parameters (tree model 2) =====

Tree Model 2
{'max_depth': 10, 'min_impurity_decrease': 0.0, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0}

Tree Model 2a
{'max_depth': 5, 'min_impurity_decrease': 0.0, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0}

Tree Model 2b
{'max_depth': 10, 'min_impurity_decrease': 0.0, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0}


In [15]:
# Create summary tables for Model 1
cv_summary_model1_df = make_cv_summary(tree_model1_results)
test_summary_model1_df = make_test_summary(tree_model1_results)

print("\n===== CV Summary Table (tree model 1) =====")
print(cv_summary_model1_df.to_string(index=False))

print("\n===== Test Summary Table (tree model 1) =====")
print(test_summary_model1_df.to_string(index=False))

cv_summary_model1_df.to_csv(
    "../results/tree_model1_cv_summary.csv",
    index=False
)

test_summary_model1_df.to_csv(
    "../results/tree_model1_test_summary.csv",
    index=False
)


===== CV Summary Table (tree model 1) =====
        Model           AUC     Precision        Recall      Accuracy            F1
 Tree Model 1 0.876 (0.001) 0.827 (0.004) 0.841 (0.003) 0.819 (0.002) 0.834 (0.001)
Tree Model 1a 0.766 (0.005) 0.496 (0.006) 0.692 (0.009) 0.739 (0.004) 0.578 (0.006)
Tree Model 1b 0.816 (0.002) 0.878 (0.001) 0.851 (0.004) 0.812 (0.002) 0.865 (0.002)

===== Test Summary Table (tree model 1) =====
        Model   AUC  Precision  Recall  Accuracy    F1
 Tree Model 1 0.873      0.827   0.834     0.816 0.830
Tree Model 1a 0.781      0.508   0.693     0.745 0.586
Tree Model 1b 0.828      0.877   0.839     0.804 0.858


In [16]:
# Create summary tables for Model 2
cv_summary_model2_df = make_cv_summary(tree_model2_results)
test_summary_model2_df = make_test_summary(tree_model2_results)

print("\n===== CV Summary Table (tree model 2) =====")
print(cv_summary_model2_df.to_string(index=False))

print("\n===== Test Summary Table (tree model 2) =====")
print(test_summary_model2_df.to_string(index=False))

cv_summary_model2_df.to_csv(
    "../results/tree_model2_cv_summary.csv",
    index=False
)

test_summary_model2_df.to_csv(
    "../results/tree_model2_test_summary.csv",
    index=False
)


===== CV Summary Table (tree model 2) =====
        Model           AUC     Precision        Recall      Accuracy            F1
 Tree Model 2 0.768 (0.001) 0.772 (0.002) 0.840 (0.004) 0.734 (0.000) 0.804 (0.000)
Tree Model 2a 0.729 (0.003) 0.703 (0.004) 0.677 (0.009) 0.678 (0.003) 0.690 (0.004)
Tree Model 2b 0.754 (0.007) 0.858 (0.005) 0.744 (0.008) 0.725 (0.004) 0.797 (0.004)

===== Test Summary Table (tree model 2) =====
        Model   AUC  Precision  Recall  Accuracy    F1
 Tree Model 2 0.771      0.771   0.842     0.735 0.805
Tree Model 2a 0.734      0.700   0.715     0.685 0.707
Tree Model 2b 0.760      0.856   0.730     0.717 0.788
